# Version 2 — Improved Features and Model

This notebook documents the Version 2 development process: what changed from V1, why, and what the results look like.

**V2 improvements over V1:**
- Expanded Elo code map (48 → 108 teams) — halves Elo null rate
- Competitive matches only for training — removes friendly noise
- `match_importance` feature (score 1–5 by tournament type)
- `h2h_win_rate` and `h2h_goal_diff` features (last 10 meetings)
- XGBoost hyperparameter tuning with `RandomizedSearchCV` + `TimeSeriesSplit`

**Scripts:**
- `src/feature_engineering/build_features_v2.py` → `data/processed/features_v2.csv`
- `src/models/train_model_v2.py` → `models/best_model_v2.pkl`
- `src/simulation/simulate_tournament_v2.py` → `data/processed/simulation_results.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

---
## 1. Feature Dataset — V1 vs V2

In [ ]:
features_v1 = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])
features_v2 = pd.read_csv('../data/processed/features_v2.csv', parse_dates=['date'])

print('V1 features shape:', features_v1.shape)
print('V2 features shape:', features_v2.shape)
print()
print('New columns in V2:', sorted(set(features_v2.columns) - set(features_v1.columns)))

In [ ]:
# Compare Elo null rates
v1_elo_null = features_v1['home_elo_rating'].isna().mean()
v2_elo_null = features_v2['home_elo_rating'].isna().mean()

print(f'Elo null rate — V1: {v1_elo_null:.1%}   V2: {v2_elo_null:.1%}')
print(f'Improvement: {v1_elo_null - v2_elo_null:.1%} fewer nulls')

In [ ]:
# Show null rate for every feature in V2
null_rates = features_v2.isnull().mean().sort_values(ascending=False)
null_rates_pct = (null_rates * 100).round(1)
print('Null rates in features_v2.csv:')
print(null_rates_pct[null_rates_pct > 0].to_string())

---
## 2. Match Importance Distribution

In [ ]:
# New in V2: match_importance score
importance_map = {1: 'Friendly', 2: 'Competitive', 3: 'Qualifier', 4: 'Continental Tourn.', 5: 'World Cup'}
imp_counts = features_v2['match_importance'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    [importance_map.get(i, str(i)) for i in imp_counts.index],
    imp_counts.values,
    color=['#cccccc', '#aec6cf', '#7fb3d3', '#3a7abf', '#1a3a6b']
)
for bar, val in zip(bars, imp_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=9)
ax.set_title('Match Importance Score Distribution (V2 feature set)', fontweight='bold')
ax.set_ylabel('Number of matches')
ax.set_xlabel('Match type')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

print('\nCounts by importance score:')
for score, count in imp_counts.items():
    print(f'  {score} ({importance_map.get(score, "?")}): {count:,} matches ({count/len(features_v2):.1%})')

In [ ]:
# Home win rate by match importance — does importance matter?
home_win_by_imp = features_v2.groupby('match_importance')['result'].apply(
    lambda x: (x == 'home_win').mean()
).rename('home_win_rate')

draw_rate_by_imp = features_v2.groupby('match_importance')['result'].apply(
    lambda x: (x == 'draw').mean()
).rename('draw_rate')

away_win_by_imp = features_v2.groupby('match_importance')['result'].apply(
    lambda x: (x == 'away_win').mean()
).rename('away_win_rate')

outcome_by_imp = pd.concat([home_win_by_imp, draw_rate_by_imp, away_win_by_imp], axis=1)
outcome_by_imp.index = [importance_map.get(i, str(i)) for i in outcome_by_imp.index]

print('Outcome distribution by match importance:')
print((outcome_by_imp * 100).round(1).to_string())

---
## 3. Training Data: All Matches vs Competitive Only

In [ ]:
# V2 trains on competitive matches only (importance > 1)
v2_competitive = features_v2[features_v2['match_importance'] > 1]
v2_train = v2_competitive[v2_competitive['date'].dt.year < 2018]
v2_test  = v2_competitive[v2_competitive['date'].dt.year >= 2018]

v1_train = features_v1[features_v1['date'].dt.year < 2018]
v1_test  = features_v1[features_v1['date'].dt.year >= 2018]

print('Training set comparison:')
print(f'  V1 train (all matches):       {len(v1_train):>6,} rows')
print(f'  V2 train (competitive only):  {len(v2_train):>6,} rows')
print(f'  Rows removed (friendlies):    {len(v1_train) - len(v2_train):>6,} rows')
print()
print('Test set comparison:')
print(f'  V1 test:  {len(v1_test):>6,} rows')
print(f'  V2 test:  {len(v2_test):>6,} rows')

---
## 4. Model Comparison — V2

In [ ]:
model_comp_v2 = pd.read_csv('../data/processed/model_comparison_v2.csv')
model_comp_v1 = pd.read_csv('../data/processed/model_comparison.csv')

print('V2 Model Comparison:')
print(model_comp_v2.to_string(index=False))

In [ ]:
# Side-by-side bar chart: V1 best vs V2 best on all metrics
v1_best = model_comp_v1.loc[model_comp_v1['log_loss'].idxmin()]
v2_best = model_comp_v2.loc[model_comp_v2['log_loss'].idxmin()]

metrics = ['accuracy', 'f1', 'roc_auc']
labels  = ['Accuracy', 'F1 Score', 'ROC-AUC']

v1_vals = [v1_best[m] for m in metrics]
v2_vals = [v2_best[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, v1_vals, width, label=f'V1 ({v1_best["model"]})', color='#aec6cf')
bars2 = ax.bar(x + width/2, v2_vals, width, label=f'V2 ({v2_best["model"]})', color='#1a3a6b')

for bar, val in zip(bars1, v1_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9, color='#333')
for bar, val in zip(bars2, v2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9, color='#1a3a6b', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0.45, 0.85)
ax.set_title('V1 vs V2 — Best Model Comparison', fontweight='bold')
ax.set_ylabel('Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Log loss comparison (lower is better)
v1_ll = v1_best['log_loss']
v2_ll = v2_best['log_loss']

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['V1 XGBoost', 'V2 XGBoost (tuned)'], [v1_ll, v2_ll],
              color=['#aec6cf', '#1a3a6b'], width=0.4)
for bar, val in zip(bars, [v1_ll, v2_ll]):
    ax.text(bar.get_x() + bar.get_width()/2, val - 0.01, f'{val:.4f}',
            ha='center', va='top', fontsize=11, color='white', fontweight='bold')
ax.set_title('Log Loss — V1 vs V2 (lower is better)', fontweight='bold')
ax.set_ylabel('Log Loss')
ax.set_ylim(0.82, 0.92)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Log loss improvement: {v1_ll:.4f} → {v2_ll:.4f}  (−{(v1_ll - v2_ll):.4f}, −{(v1_ll - v2_ll)/v1_ll:.1%})')

---
## 5. V2 Tournament Simulation Results

In [ ]:
sim = pd.read_csv('../data/processed/simulation_results.csv')
print('2026 FIFA World Cup — V2 Championship Probabilities (10,000 simulations)')
print(sim.to_string(index=False))

In [ ]:
# Championship probability bar chart
sim_sorted = sim.sort_values('win_pct', ascending=True)

colors = ['#d4edda' if p < 5 else '#7fb3d3' if p < 10 else '#1a3a6b'
          for p in sim_sorted['win_pct']]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(sim_sorted['team'], sim_sorted['win_pct'], color=colors)

for bar, val in zip(bars, sim_sorted['win_pct']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.set_xlabel('Championship Probability (%)')
ax.set_title('2026 FIFA World Cup — V2 Model Championship Probabilities\n(10,000 Monte Carlo simulations from Round of 16)', fontweight='bold')
ax.set_xlim(0, 24)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# V1 vs V2 simulation comparison — top 8
sim_v1 = pd.DataFrame({
    'team':    ['France', 'Brazil', 'Spain', 'Argentina', 'Morocco',
                'England', 'Belgium', 'Portugal', 'Mexico'],
    'win_pct_v1': [21.8, 13.5, 13.4, 11.3, 7.6, 7.1, 7.1, 7.0, 4.8]
})

comparison = sim[['team', 'win_pct']].merge(sim_v1, on='team', how='inner')
comparison['delta'] = comparison['win_pct'] - comparison['win_pct_v1']
comparison = comparison.sort_values('win_pct', ascending=False)

print('V1 vs V2 simulation — championship probability shift:')
print(comparison.to_string(index=False))

In [ ]:
# Waterfall-style delta chart
comp_plot = comparison.sort_values('delta')
colors_delta = ['#e74c3c' if d < 0 else '#27ae60' for d in comp_plot['delta']]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(comp_plot['team'], comp_plot['delta'], color=colors_delta)
ax.axvline(0, color='black', linewidth=0.8)

for i, (val, team) in enumerate(zip(comp_plot['delta'], comp_plot['team'])):
    ax.text(val + (0.1 if val >= 0 else -0.1), i,
            f'{val:+.1f}pp', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)

ax.set_xlabel('Change in championship probability (V2 − V1)')
ax.set_title('How V2 Changed Predictions vs V1', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Summary

| Metric | V1 | V2 | Δ |
|---|---|---|---|
| Accuracy | 59.18% | 62.08% | **+2.90pp** |
| F1 Score | 0.5185 | 0.5427 | **+0.0242** |
| ROC-AUC | 0.7467 | 0.7752 | **+0.0285** |
| Log Loss | 0.8863 | 0.8510 | **−4.0%** |

### Key takeaways

1. **Filtering out friendlies was the biggest single improvement.** Removing 28,165 low-signal matches tightened the training set and produced the largest accuracy gain.

2. **Elo map quality mattered more than Elo coverage.** The 70% V1 null rate was caused almost entirely by name mismatches, not missing data. Fixing the map halved nulls without new data sources.

3. **Match importance improved probability calibration.** Log loss (which measures calibration) dropped by 4%, more than accuracy improved proportionally.

4. **Morocco rose sharply (+3.6pp).** V2's H2H and match importance features gave Morocco more credit for their 2022 World Cup semi-final run, which V1 underweighted.

5. **Brazil fell sharply (−5.7pp).** Much of Brazil's V1 advantage came from friendly matches boosting their rolling form. Competitive-only training corrected this.

**Model's pick: France at 19.8% championship probability.**